# llm_grid

> Phase 10 grids (pilot, main, Sonnet, equal budget, framing, memory, persistence, liar, numpy, quick), the cost estimate per grid from the token profile, and the analysis plan with its hash (frozen after the pilot, before the main runs).

In [ ]:
#| default_exp llm_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import copy
import hashlib
import json

from explRL.llm_client import PRICES, TOKEN_PROFILE, estimateCost
from explRL.llm_compare import runNumpyCell
from explRL.llm_runs import LLMConfig, runLLMCell

CHANNELS, SIZES, TASKS = ('none', 'blackboard', 'messages', 'hierarchical'), (1, 2, 4, 8), ('forage', 'board')
HAIKU, SONNET = 'claude-haiku-5-5', 'claude-sonnet-5-5'
SETTINGS = dict(effort='low', maxTokens=8000, difficulty=8, solveRate=0.889,   # fixed by the pilot (T7, 2026-10-12)
                forageWorld=dict(tStay=5.0))     # 5 s stays: 50 turns span several 40 s patch moves (user, 2026-10-12)
ANALYSIS_PLAN = dict(
    version=1, frozen=True, models=dict(main=HAIKU, sizeContrast=SONNET), settings=SETTINGS, prices=PRICES,
    seeds=dict(main=list(range(500, 505)), sonnet=list(range(500, 503)), budget=list(range(600, 603)),
               framing=list(range(610, 613)), memory=list(range(610, 613)), persist=list(range(620, 623)),
               liar=list(range(630, 633))),
    metrics=['welfare', 'welfareHonest', 'soShare', 'discoveries', 'invalid', 'reportShare', 'exploreShare',
             'exploreSpread', 'freeRiding', 'diversity', 'rmrt', 'ifdMean', 'solved', 'duplicates', 'wrong',
             'inputTokens', 'outputTokens', 'cost', 'cacheHit'],
    analyses=dict(
        main='welfare per agent-turn vs N per channel and task (Haiku, seeds 500-504)',
        harnessBenefit='per (channel, N): welfare minus the none cell, paired on seed',
        sizeContrast='board, N 1 and 4: harness benefit Sonnet minus Haiku, paired on seeds 500-502; mean, bootstrap CI',
        numpy='Spearman rho over the 16 (channel, N) cells per task; 95% bootstrap CI resampling seeds',
        tokens='team welfare vs total tokens per run; equal budget: welfare per agent-turn vs N at B = 200',
        arms='framing, persistence, memory, liar: difference from the control cell, bootstrap CI; liar arm: sign of '
             'blackboard-minus-messages loss compared with H3'),
    nboot=2000, ci=0.95, exploratory=True)


def planHash(plan=ANALYSIS_PLAN):
    return hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()[:12]


def llmCell(study, factor, level, N, seed, runner='llm', **cfg):
    return dict(study=study, factor=factor, level=str(level), N=N, seed=seed, runner=runner,
                cfg=LLMConfig(N=N, **cfg).asDict())


def _world(task, runner, s):
    if task == 'forage':
        return dict(s['forageWorld'])
    return dict(solveRate=s['solveRate']) if runner == 'numpy' else {}


def _main(study, model, sizes, seeds, tasks=TASKS, runner='llm', s=SETTINGS):
    return [llmCell(study, 'main', t, N, sd, runner, task=t, channel=ch, model=model, effort=s['effort'],
                    maxTokens=s['maxTokens'], difficulty=s['difficulty'], world=_world(t, runner, s))
            for t in tasks for ch in CHANNELS for N in sizes for sd in seeds]


def mainGrid(s=SETTINGS):
    return _main('main', HAIKU, SIZES, range(500, 505), s=s)


def sonnetGrid(s=SETTINGS):
    return _main('sonnet', SONNET, (1, 4), range(500, 503), ('board',), s=s)


def numpyGrid(task, s=SETTINGS):
    "The main-grid cells of `task` for the numpy teams (decision L12)."
    return _main('numpy', HAIKU, SIZES, range(500, 505), (task,), 'numpy', s=s)


def _arm(study, levels, seeds, s=SETTINGS, **fixed):
    kw = dict(effort=s['effort'], maxTokens=s['maxTokens'], difficulty=s['difficulty'])
    out = [llmCell(study, study, lv, cfg.get('N', 4), sd,
                   **{**kw, 'channel': 'blackboard', **fixed, **{k: v for k, v in cfg.items() if k != 'N'}})
           for lv, cfg in levels for sd in seeds]
    for c in out:
        c['cfg']['world'] = _world(c['cfg']['task'], 'llm', s)
    return out


def budgetGrid(s=SETTINGS):
    return _arm('budget', [(f'{ch}|N={N}', dict(channel=ch, N=N, B=200)) for ch in ('none', 'blackboard') for N in SIZES],
                range(600, 603), s)


def framingGrid(s=SETTINGS):
    return _arm('framing', [(f, dict(framing=f)) for f in ('own', 'team', 'credit')], range(610, 613), s)


def memoryGrid(s=SETTINGS):
    return _arm('memory', [(m, dict(memory=m)) for m in ('last8', 'notes')], range(610, 613), s)


def persistGrid(s=SETTINGS):
    return _arm('persist', [(f'lessons={int(x)}', dict(lessons=x)) for x in (False, True)], range(620, 623), s,
                episodes=4, turns=25)


def liarGrid(s=SETTINGS):
    return _arm('liar', [(f'{t}|{ch}|liar={n}', dict(task=t, channel=ch, nLiar=n)) for t in TASKS
                         for ch in ('blackboard', 'messages') for n in (0, 1)], range(630, 633), s)


def pilotGrid(stage=1, difficulty=None, levels=(6, 8), s=SETTINGS):
    """Stage 1: board solo solve rate at the difficulty `levels`, and foraging; stage 2 (at the chosen difficulty):
    effort and Sonnet. All at the settings' `maxTokens` and foraging world."""
    kw = dict(turns=30, maxTokens=s['maxTokens'])
    if stage == 1:
        return [llmCell('pilot', 'difficulty', f'board|d={d}', 1, sd, task='board', difficulty=d, **kw)
                for d in levels for sd in range(1000, 1005)] + \
               [llmCell('pilot', 'forage', f'forage|{ch}', N, 1010, channel=ch, world=dict(s['forageWorld']), **kw)
                for ch, N in (('none', 1), ('blackboard', 4))]
    return [llmCell('pilot', 'effort', 'board|medium', 1, sd, task='board', difficulty=difficulty, effort='medium',
                    **kw) for sd in range(1000, 1005)] + \
           [llmCell('pilot', 'sonnet', 'board|sonnet', 1, sd, task='board', difficulty=difficulty, model=SONNET, **kw)
            for sd in range(1000, 1003)]


GRIDS = dict(main=mainGrid, sonnet=sonnetGrid, budget=budgetGrid, liar=liarGrid, framing=framingGrid,
             memory=memoryGrid, persist=persistGrid)          # priority order for the main runs (T9)


def quickGrid():
    "Every runner path at 3 turns (fake client): the main cells at one seed, the first seed of every other grid, numpy."
    out = []
    for name, g in GRIDS.items():
        cells = g()
        first = cells[0]['seed']
        out += [c for c in cells if c['seed'] == first]
    out += [c for t in TASKS for c in numpyGrid(t) if c['seed'] == 500]
    out = copy.deepcopy(out)
    for c in out:
        cfg = c['cfg']
        cfg['turns'], cfg['B'] = (2 if cfg['episodes'] > 1 else 3), (cfg['N'] * 3 if cfg['B'] else 0)
        cfg['episodes'] = min(cfg['episodes'], 2)
    return out


def withClient(cells, spec, logDir=None):
    "Attach the client spec (fake or live, cache, cap, ledger) and the turn-log directory to the LLM cells."
    return [dict(c, client=spec, logDir=logDir) if c['runner'] == 'llm' else c for c in cells]


def runAnyCell(cell):
    return runLLMCell(cell) if cell['runner'] == 'llm' else runNumpyCell(cell)


def estimateGrid(cells, profile=TOKEN_PROFILE, prices=PRICES):
    "Turn calls, belief calls and expected cost (from the token profile), per model."
    out = {}
    for c in cells:
        if c['runner'] != 'llm':
            continue
        cfg = LLMConfig.fromDict(c['cfg'])
        n, T, e = cfg.N - cfg.nLiar, cfg.turnsPerAgent(), cfg.episodes
        b = n * e if cfg.task == 'forage' else 0
        m = out.setdefault(cfg.model, dict(calls=0, beliefCalls=0, cost=0.0))
        m['calls'] += n * T * e
        m['beliefCalls'] += b
        m['cost'] += estimateCost(cfg.model, cfg.task, n, T * e, prices, profile) + \
            (estimateCost(cfg.model, cfg.task, b, 1, prices, profile) if b else 0.0)
    return out

## Tests

In [ ]:

from explRL.llm_grid import *
from explRL.llm_runs import LLMConfig

# cell counts and the design's call counts (design-low-level section 7)
def calls(cells, model=None):
    e = estimateGrid(cells)
    return sum(v['calls'] for k, v in e.items() if model in (None, k)), sum(v['beliefCalls'] for v in e.values())
assert len(mainGrid()) == 2 * 4 * 4 * 5 and calls(mainGrid()) == (30000, 300)
assert len(sonnetGrid()) == 4 * 2 * 3 and calls(sonnetGrid(), SONNET) == (3000, 0)
assert calls(budgetGrid())[0] == 4800 and calls(framingGrid())[0] == 1800 and calls(memoryGrid())[0] == 1200
assert calls(persistGrid())[0] == 2400 and calls(liarGrid())[0] == 4200
p1, p2 = pilotGrid(1), pilotGrid(2, difficulty=2)
assert calls(p1)[0] == 10 * 30 + 30 + 4 * 30 and calls(p2)[0] == 5 * 30 + 3 * 30
assert calls(pilotGrid(1, levels=(1, 2, 3, 4)))[0] == 20 * 30 + 30 + 4 * 30                 # the first pilot round
# every foraging cell (LLM and numpy, every grid) uses the 5 s stays; board cells do not
for g in [*GRIDS.values(), lambda: numpyGrid('forage'), lambda: numpyGrid('board'), lambda: p1]:
    for c in g():
        assert (c['cfg']['world'].get('tStay') == 5.0) == (c['cfg']['task'] == 'forage')
assert estimateGrid(numpyGrid('board')) == {} and len(numpyGrid('forage')) == 4 * 4 * 5
# seeds as decided (L19); the Sonnet cells reuse the Haiku worlds
assert {c['seed'] for c in mainGrid()} == set(range(500, 505)) and {c['seed'] for c in sonnetGrid()} == {500, 501, 502}
assert all(600 <= c['seed'] < 700 for g in (budgetGrid, framingGrid, memoryGrid, persistGrid, liarGrid) for c in g())
assert all(1000 <= c['seed'] < 1100 for c in p1 + p2)
# arms: the liar sits in a 4-agent team; persistence is 4 episodes of 25 turns; equal budget splits 200 agent-turns
assert {LLMConfig.fromDict(c['cfg']).turnsPerAgent() * c['N'] for c in budgetGrid()} == {200}
assert all(c['cfg']['episodes'] == 4 and c['cfg']['turns'] == 25 for c in persistGrid())
assert {(c['cfg']['nLiar'], c['N']) for c in liarGrid()} == {(0, 4), (1, 4)}
# cost estimate per grid (provisional token profile) within the cap
est = {k: estimateGrid(g()) for k, g in GRIDS.items()}
total = sum(v['cost'] for e in est.values() for v in e.values())
for k, e in est.items():
    print(f"{k:8s}", {m: (v['calls'] + v['beliefCalls'], round(v['cost'], 2)) for m, v in e.items()})
print('total $', round(total, 2))
assert 20 < total < 150
# the plan's hash is stable and changes with any setting
h = planHash()
assert h == planHash() and len(h) == 12 and planHash({**ANALYSIS_PLAN, 'version': 2}) != h

# the quick grid covers every grid and both runners at 3 turns
q = quickGrid()
assert {c['study'] for c in q} == set(GRIDS) | {'numpy'} and all(c['cfg']['turns'] <= 3 for c in q)
assert sum(c['runner'] == 'numpy' for c in q) == 32
spec = dict(kind='fake')
qc = withClient(q, spec, '/tmp/x')
assert all(('client' in c) == (c['runner'] == 'llm') for c in qc)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()